# Sesión 3 — MUBD

## Modelo lineal: de los datos a la predicción

**Pregunta:** ¿cómo cambia la resistencia del hormigón (`Strength`) con su composición y su edad? Seguiremos un mismo conjunto de datos desde la exploración hasta la evaluación con observaciones que el modelo no ha visto. El recorrido es: **describir → ajustar → seleccionar → diagnosticar → transformar → interpretar → evaluar en test**. Un buen ajuste en entrenamiento no implica necesariamente buenas predicciones fuera de él.

En VS Code, selecciona un kernel de Python y ejecuta las celdas **en orden** con **Shift + Enter**. Abre **Variables** para inspeccionar tablas y **Esquema / Outline** para navegar por apartados. Cada celda depende de las definiciones anteriores; si aparece una variable no definida, ejecuta las celdas previas. Los gráficos se generan al ejecutar sus celdas; cambia los argumentos de las funciones y vuelve a ejecutarlas para comparar escenarios.

Para repetir todo el análisis, usa **Reiniciar** y **Ejecutar todo**. [Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

Los **microejercicios** son prácticas breves: completa los huecos reutilizando funciones y patrones que ya aparecen en las celdas anteriores; no hace falta buscar funciones nuevas. Los ejercicios 3.1 y 3.2 dejan más tiempo para aplicar el procedimiento completo. Conserva `datos`, los modelos y las predicciones originales: las celdas siguientes los reutilizan. El modelo final se elige con entrenamiento; el test se reserva para la evaluación.

### Preparación

Importamos NumPy para cálculo numérico, pandas para las tablas, Matplotlib para los gráficos, statsmodels para el ajuste y los diagnósticos, SciPy para contrastes y transformaciones, y `threadpoolctl` para limitar los hilos durante la selección. Más adelante se usa `ipywidgets` para explorar una predicción individual.

Tres algoritmos especialmente largos se mantienen en scripts junto al cuaderno: `s3_seleccion_atras.py` (selección por AIC), `s3_diagnosticos.py` (panel de diagnósticos) y `s3_logverosimilitud_boxcox.py` (log-verosimilitud Box–Cox). El notebook los importa y explica su objetivo e interpretación; no hace falta recorrer su implementación línea por línea. La función auxiliar `explorar_prediccion`, que genera la vista HTML de un caso de test, está en `s3_explorar_prediccion.py`; el notebook conserva la conexión con el widget.

**Antes de ejecutar:** prepara el entorno según la guía de la raíz del proyecto y selecciona en VS Code el **kernel de ese mismo entorno**. Instalar una biblioteca en otro intérprete no resuelve errores de importación en este cuaderno. La configuración de Matplotlib fija un tamaño inicial de figura; cada gráfico puede indicar otro tamaño si necesita más espacio.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display
from pandas.plotting import scatter_matrix
from scipy import stats
from scipy.optimize import minimize_scalar
from statsmodels.nonparametric.smoothers_lowess import lowess
from threadpoolctl import threadpool_limits  # Se usa para limitar hilos al calcular la tabla de influencia.

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (16, 8), "figure.dpi": 100,
                     "axes.titlesize": 20, "axes.labelsize": 16,
                     "xtick.labelsize": 14, "ytick.labelsize": 14})
pd.set_option("display.max_columns", 20)

## Lectura de datos y descriptiva

Leemos el archivo de entrenamiento con `sep="\t"`: los campos están separados por tabuladores y la primera fila contiene sus nombres. El bucle busca la carpeta de datos desde el directorio actual hacia sus padres; así no depende de ejecutar el cuaderno desde una única ruta. Si no encuentra el archivo, muestra un error explícito.

Tras cargarlo, asignamos etiquetas de fila de 1 a $n$ y el nombre `observacion`. **Son identificadores para localizar casos, no fechas ni un orden de recogida demostrado.** Guardamos los nombres originales antes de crear variables nuevas: `Strength` será la respuesta $y$ y las ocho columnas restantes serán las predictoras $X$. Más adelante esas listas permiten comparar y mostrar los datos originales sin mezclar transformaciones.

In [ ]:
DATA_PATH = Path.cwd().parent.parent / "datos"

datos = pd.read_csv(DATA_PATH / "Concrete_train.txt", sep="\t")
datos.index = np.arange(1, len(datos) + 1)
datos.index.name = "observacion"
columnas_originales = list(datos.columns)
predictores = [columna for columna in datos if columna != "Strength"]

### Inspección inicial

Ejecuta las tres celdas siguientes en orden:

1. `datos.shape` devuelve **(filas, columnas)**: comprueba cuántos casos y variables tenemos antes de ajustar modelos.
2. `datos.head()` muestra las primeras filas para revisar nombres, unidades y valores; `datos.loc[[5]]` selecciona la observación etiquetada 5, no la quinta posición por otra convención.
3. `datos.describe()` resume, para cada variable numérica, recuento, media, desviación típica, mínimo, cuartiles y máximo. Compara escalas y rangos; el recuento puede revelar datos faltantes, pero un resumen agregado no muestra todas las relaciones entre variables.

**Antes de modelizar**, pregúntate qué columna queremos predecir y si las variables de entrada podrían contener valores extremos o relaciones no lineales.

In [ ]:
datos.shape  # Número de filas y columnas

In [ ]:
datos.head()

In [ ]:
datos.describe()

#### Diagramas de caja de todas las variables

Cada caja va del primer al tercer cuartil y la línea interior marca la mediana; los bigotes y los puntos separados ayudan a detectar observaciones alejadas del grueso. Los gráficos se dibujan **en la escala original de cada columna**, por lo que una variable con valores numéricamente grandes puede comprimir visualmente otras cajas: no compares alturas entre variables con unidades o rangos distintos como si midieran lo mismo.

Un valor que parece atípico no es automáticamente un error. Anótalo para revisarlo junto con el resto de variables y los diagnósticos del modelo. Giramos las etiquetas del eje horizontal para poder leer todos los nombres.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
_ = ax.boxplot([datos[columna].dropna() for columna in columnas_originales],
               patch_artist=True,
               boxprops={"facecolor": "#9fc3e4", "edgecolor": "#666666", "linewidth": 1.2},
               whiskerprops={"color": "#666666"}, capprops={"color": "#666666"},
               medianprops={"color": "#800000", "linewidth": 1.8})
_ = ax.set_xticks(np.arange(1, len(columnas_originales) + 1))
_ = ax.set_xticklabels(columnas_originales, rotation=45, ha="right")
_ = ax.set_title("Distribución de las variables originales")
ax.tick_params(labelsize=14)
fig.tight_layout()
plt.show()

#### Explorar todos los pares de variables

La matriz de dispersión reúne un gráfico por pareja de columnas: en la diagonal hay histogramas y fuera de ella, nubes de puntos. Busca asociaciones entre `Strength` y las predictoras, posibles curvas, grupos y también relaciones fuertes **entre predictoras** que puedan dificultar la interpretación de coeficientes. El mismo par aparece reflejado a ambos lados de la diagonal, con los ejes intercambiados.

Este gráfico es **exploratorio**: una nube con forma de curva sugiere probar otra especificación, no demuestra que exista una relación causal. La transparencia evita que la superposición de muchos puntos oculte por completo su densidad.

In [ ]:
axes = scatter_matrix(datos[columnas_originales], figsize=(18, 18), diagonal="hist",
                      color="#9fc3e4", alpha=0.5, s=16,
                      hist_kwds={"color": "#9fc3e4", "edgecolor": "#666666"})
for ax in axes.flat:
    ax.tick_params(labelsize=9)
    ax.xaxis.label.set_fontsize(11)
    ax.yaxis.label.set_fontsize(11)
plt.show()

### Relación entre resistencia y cemento

Los puntos muestran cada par (`Cement`, `Strength`); la curva **LOWESS** resume la tendencia local sin obligarla a ser una recta. `fraccion` controla aproximadamente qué proporción de observaciones contribuye a cada ajuste local: una fracción menor sigue más los detalles y también el ruido, y una mayor suaviza más. Prueba con otro predictor o cambia la fracción en la llamada siguiente para comparar.

La función detecta avisos numéricos o valores no finitos y, si hace falta, amplía el vecindario en incrementos pequeños; indica en el gráfico la fracción finalmente utilizada. **Cómo leerlo:** si la pendiente cambia claramente a lo largo del eje horizontal, un único coeficiente lineal podría no describir toda la relación. LOWESS ayuda a proponer modelos; no sustituye su validación.

In [ ]:
def dibujar_relacion(ax, x, y, nombre, fraccion=2/3):
    ax.scatter(x, y, s=30, color="#9fc3e4", edgecolor="#666666",
               linewidth=0.4, alpha=0.6)
    aplicada = fraccion
    while True:
        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter("always", RuntimeWarning)
            curva = lowess(y, x, frac=aplicada, it=3, delta=0.01 * np.ptp(x))
        numericos = any(issubclass(aviso.category, RuntimeWarning) for aviso in avisos)
        if np.isfinite(curva).all() and not numericos:
            break
        if aplicada >= 1:
            raise ValueError("No se puede estimar LOWESS con estos valores.")
        aplicada = min(1, aplicada + 0.05)
    ax.plot(curva[:, 0], curva[:, 1], color="#800000", linewidth=1.8)
    ax.set(xlabel=nombre, ylabel="Strength")
    if aplicada > fraccion + 1e-8:
        ax.text(0.02, 0.98, f"Fracción aplicada: {aplicada:.2f}",
                transform=ax.transAxes, va="top", fontsize=11)

def explorar_relacion(variable, fraccion):
    fig, ax = plt.subplots(figsize=(16, 8))
    dibujar_relacion(ax, datos[variable], datos["Strength"], variable, fraccion)
    ax.set_title("Resistencia y " + variable)
    fig.tight_layout()
    plt.show()

In [ ]:
# Cambia "Cement" por otro predictor o ajusta la fracción para comparar curvas.
explorar_relacion("Cement", 2/3)

#### Descriptiva bivariante para las ocho variables predictoras

Repetimos la nube de puntos y el suavizado LOWESS para cada predictor frente a `Strength`. Observa en qué variables la tendencia parece casi recta, en cuáles se curva y en cuáles se acumulan muchos casos en un mismo valor (por ejemplo, ceros). También importa si la dispersión vertical cambia al aumentar la variable.

**Precaución:** cada panel es una relación *marginal*, sin mantener constantes las otras siete variables. Un patrón puede cambiar al ajustar conjuntamente; no interpretes estas curvas como efectos aislados. La comparación sirve para decidir qué revisar en los residuos y si merece la pena considerar términos no lineales.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, variable in zip(axes.flat, predictores):
    dibujar_relacion(ax, datos[variable], datos["Strength"], variable)
    _ = ax.set_title(variable, fontsize=16)
    ax.xaxis.label.set_fontsize(13)
    ax.yaxis.label.set_fontsize(13)
    ax.tick_params(labelsize=11)
fig.tight_layout()
plt.show()

## Ajuste del modelo lineal simple

Primero predecimos `Strength` solo con `Cement`: $\widehat{y}=\widehat{\beta}_0+\widehat{\beta}_1\,\mathrm{Cement}$. `sm.add_constant` añade la columna de unos necesaria para estimar el intercepto; `sm.OLS(...).fit()` calcula los coeficientes por mínimos cuadrados. Con `missing="raise"` preferimos detenernos ante datos faltantes en vez de ignorar silenciosamente filas.

En el resumen, compara el signo y tamaño de la pendiente, su incertidumbre, el $R^2$ y el número de observaciones. La pendiente indica el cambio *promedio estimado* al añadir una unidad de cemento **en este modelo de una sola variable**: no aísla el efecto del cemento respecto a los demás componentes. El segundo gráfico superpone los datos y la recta predicha para ver dónde se aparta de la nube; se predice sobre una rejilla dentro del rango observado, sin extrapolar.

In [ ]:
y = datos["Strength"]
X_simple = sm.add_constant(datos[["Cement"]], has_constant="add")
mod_lm0 = sm.OLS(y, X_simple, missing="raise").fit()
mod_lm0.summary()

In [ ]:
# Ejemplo: tres residuos de mayor magnitud en el modelo simple.
mod_lm0.resid.abs().nlargest(3)

In [ ]:
rejilla = pd.DataFrame({"Cement": np.linspace(datos["Cement"].min(), datos["Cement"].max(), 100)})
fig, ax = plt.subplots(figsize=(16, 8))
_ = ax.scatter(datos["Cement"], y, s=40, color="#9fc3e4", alpha=0.65,
               edgecolor="#666666", linewidth=0.6)
_ = ax.plot(rejilla["Cement"], mod_lm0.predict(sm.add_constant(rejilla, has_constant="add")),
            color="#800000", linewidth=1.8)
_ = ax.set(xlabel="Cement", ylabel="Strength", title="Modelo lineal simple")
fig.tight_layout()
plt.show()

## Ajuste del modelo multivariable

Incluimos las ocho predictoras reales y, **solo en `X_lineal`**, añadimos dos columnas de ruido artificial. El generador con semilla 42 reproduce exactamente los mismos valores al volver a ejecutar. No se añaden a `datos` ni a `predictores`, por lo que no aparecen después en los modelos polinómicos. `const` es el intercepto; `mod_lm1.params` muestra los coeficientes y `mod_lm1.summary()` añade errores estándar, contrastes e indicadores de ajuste.

Aquí cada coeficiente describe el cambio estimado en la respuesta al variar **una** predictora, manteniendo constantes las demás columnas del modelo. Si estas están relacionadas, la estimación puede volverse inestable. Añadir columnas no reduce el $R^2$ en **entrenamiento** por construcción, pero puede empeorar la predicción en datos nuevos. Los términos de ruido ilustran por qué un ajuste aparentemente mejor no basta para decidir qué conservar. Los p-valores dependen del modelo completo: interprétalos con cautela, no como criterio único de selección.

In [ ]:
# Construye el diseño multivariable y añade dos predictoras de ruido reproducibles.
azar = np.random.default_rng(42)
X_lineal = sm.add_constant(datos[predictores], has_constant="add")
X_lineal["Ruido1"] = azar.standard_normal(len(datos))
X_lineal["Ruido2"] = azar.standard_normal(len(datos))
mod_lm1 = sm.OLS(y, X_lineal, missing="raise").fit()
# Para probar una transformación sin alterar esta matriz, trabaja con X_lineal.copy().

In [ ]:
mod_lm1.summary()

In [ ]:
# Ejemplo de patrón: quitar la constante y ordenar los coeficientes de menor a mayor.
mod_lm1.params.drop(index="const").sort_values(ascending=True)

**Microejercicio — selección manual (primer paso).** Usa el patrón de la celda anterior, cambiando los coeficientes por los p-valores y el orden a mayor primero. ¿Qué variable revisarías antes para una posible eliminación? Compara después tu candidata con el primer paso de la selección por AIC. Un p-valor alto **no basta por sí solo para eliminarla**: también habría que comprobar cómo cambia el ajuste y si afecta a los coeficientes de las otras variables.

In [ ]:
try:
    # Ordena los p-valores de las predictoras, excluyendo el término constante.
    p_sin_constante = mod_lm1.pvalues.drop(index=...)  # Completa el nombre del término constante.
    p_ordenados = p_sin_constante.sort_values(ascending=...)  # Elige el orden para que el p-valor mayor aparezca primero.
    print(p_ordenados)  # Identifica qué predictor revisarías antes y compáralo con el primer paso AIC.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Selección de variables: eliminación hacia atrás con AIC

Partimos del modelo multivariable con ocho predictoras reales y dos de ruido. En cada iteración, el procedimiento ajusta modelos que retiran **un grupo completo de términos**. Elige la retirada con el AIC menor y solo la acepta si reduce el AIC actual; termina cuando ninguna retirada lo mejora. El AIC combina ajuste y penalización por complejidad: **menor es mejor al comparar modelos con la misma respuesta y muestra**.

La lógica está encapsulada en `seleccion_atras`, importada desde `s3_seleccion_atras.py`. Recibe la respuesta, la matriz de diseño y los grupos de columnas que deben mantenerse juntos; devuelve el modelo elegido, los términos activos y el historial de pasos. Aquí nos centramos en el criterio de selección y en interpretar la tabla, no en recorrer la implementación línea por línea.

La tabla `pasos_lineales` muestra el estado inicial, qué término se eliminó en cada paso, el nuevo AIC y cuántos términos quedan. Observa qué ocurre con `Ruido1` y `Ruido2`: no se fuerza su eliminación solo por llamarse ruido. El proceso es exploratorio y depende de la muestra; un AIC menor no garantiza que se cumplan los supuestos, y los p-valores del modelo elegido no incorporan la incertidumbre de la selección.

In [ ]:
from s3_seleccion_atras import seleccion_atras  # Importa el procedimiento de selección por AIC encapsulado en su script.

In [ ]:
grupos_lineales = {variable: [variable] for variable in X_lineal.columns if variable != "const"}
mod_lm2, seleccion_lineal, pasos_lineales = seleccion_atras(y, X_lineal, grupos_lineales)
pasos_lineales

#### Microejercicio práctico — medir el cambio del AIC

Usa los modelos anterior y seleccionado (`mod_lm1` y `mod_lm2`) para calcular cuánto disminuyó el AIC y qué términos desaparecieron. Completa los dos huecos con operaciones sobre las variables ya creadas; ejecuta y verifica el primer término eliminado en `pasos_lineales`.

In [ ]:
try:
    # Compara el AIC antes y después de la selección y lista los términos retirados.
    mejora_aic = ...  # Resta los atributos .aic de mod_lm1 y mod_lm2.
    terminos_retirados = ...  # Usa una lista por comprensión sobre grupos_lineales y seleccion_lineal, como en el bloque anterior.
    print(f'Reducción de AIC: {mejora_aic:.2f}')  # Un valor positivo indica una reducción del AIC.
    print('Términos retirados:', terminos_retirados)  # Comprueba qué predictores desaparecieron.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

In [ ]:
mod_lm2.summary()

## Validación del modelo lineal

Antes de confiar en `mod_lm2`, inspeccionamos cuatro vistas de sus residuos $e_i=y_i-\widehat y_i$:

1. **Residuos frente a ajustados:** buscamos una banda sin curva clara alrededor de cero; una tendencia LOWESS puede sugerir que falta forma funcional.
2. **Gráfico Q–Q:** compara los cuantiles de los residuos estandarizados con una normal; desviaciones, sobre todo en las colas, aconsejan revisar la aproximación.
3. **Escala y localización:** representa $\sqrt{|e_i^*|}$ frente al valor ajustado; una pauta creciente o decreciente puede indicar varianza no constante.
4. **Residuos frente a apalancamiento:** ayuda a localizar combinaciones de predictores poco habituales y casos potencialmente influyentes. Las curvas de Cook son referencias, no reglas de eliminación.

La función `diagnosticos`, importada desde `s3_diagnosticos.py`, reúne estos cuatro gráficos a partir de un modelo ya ajustado. La interpretación se centra en los patrones y sus supuestos, no en cómo programar cada panel.

**No son pruebas automáticas de validez.** Los gráficos requieren inspección contextual; la independencia de los errores se aborda aparte y no puede inferirse de estos paneles.

In [ ]:
from s3_diagnosticos import diagnosticos  # Importa la función que genera los cuatro gráficos de diagnóstico.

In [ ]:
diagnosticos(mod_lm2, "Diagnóstico del modelo lineal seleccionado")

### Independencia de los errores

Dibujamos los residuos en el **orden de las filas del archivo de entrenamiento**. Una racha larga del mismo signo, una tendencia o un ciclo podrían señalar estructura que el modelo no recoge. La línea horizontal a cero facilita comparar signos y amplitudes.

**Límite importante:** el índice que hemos asignado no garantiza orden cronológico ni espacial. Si las filas se mezclaron antes de recibirlas, este gráfico no permite comprobar dependencia temporal; tampoco la ausencia de un patrón visible demuestra independencia. Para estudiarla rigurosamente harían falta datos sobre el muestreo y, si procede, el orden real de observación.

In [ ]:
def grafico_orden_residuos(modelo, titulo):
    fig, ax = plt.subplots(figsize=(16, 6))
    orden = np.arange(1, len(modelo.resid) + 1)
    _ = ax.plot(orden, modelo.resid, color="#34699a", alpha=0.7, linewidth=1.1,
                marker=".", markersize=4)
    _ = ax.axhline(0, color="#800000", linestyle="--", linewidth=1.8)
    _ = ax.set(xlabel="Orden de las filas en entrenamiento", ylabel="Residuo", title=titulo)
    fig.tight_layout()
    plt.show()

grafico_orden_residuos(mod_lm2, "Residuos del modelo lineal en el orden de las filas")

**Microejercicio — residuos extremos.** Reutiliza el patrón del modelo simple mostrado antes: identifica las tres observaciones con mayor residuo en valor absoluto de `mod_lm2`. ¿Por qué conviene examinarlas antes de decidir si son errores?

In [ ]:
try:
    # Encuentra las tres observaciones con residuos de mayor magnitud en el modelo lineal.
    residuos_abs = mod_lm2.resid.abs()  # Convierte cada residuo en su magnitud.
    top_residuos = residuos_abs.nlargest(...)  # Completa el número de observaciones que vas a listar.
    print(top_residuos)  # Muestra las etiquetas de fila y el tamaño de esos residuos.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Nueva descriptiva: lo que aún no explica el modelo

### Residuos frente a las variables predictoras

Ahora repetimos los gráficos exploratorios con los **residuos del modelo seleccionado** en el eje vertical, no con `Strength`. Si el modelo recogiera adecuadamente la relación con cada predictor, los residuos no deberían mostrar una curva persistente alrededor de la línea horizontal de cero. LOWESS ayuda a ver tendencias difíciles de apreciar en una nube densa.

Un patrón aquí puede indicar curvatura omitida, un grupo distinto o una transformación necesaria. **No significa necesariamente causalidad** ni justifica modificar a ciegas el modelo: conviene verificar si hay valores extremos, correlación entre predictoras y estabilidad de la tendencia. Compararemos estas gráficas con las del modelo polinómico.

In [ ]:
def residuos_predictoras(modelo, tabla, variables):
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, variable in zip(axes.flat, variables):
        dibujar_relacion(ax, tabla[variable], modelo.resid, variable)
        ax.set(title=variable, ylabel="Residuos")
        ax.axhline(0, color="#666666", linestyle="--", linewidth=1.2)
    for ax in list(axes.flat)[len(variables):]:
        ax.set_visible(False)
    for ax in axes.flat:
        if ax.get_visible():
            ax.title.set_fontsize(16)
            ax.xaxis.label.set_fontsize(13)
            ax.yaxis.label.set_fontsize(13)
            ax.tick_params(labelsize=11)
    fig.tight_layout()
    plt.show()

residuos_predictoras(mod_lm2, datos, predictores)

### Gráficos de residuos y pruebas de curvatura

La función devuelve **gráficos y una tabla de pruebas F**. Para cada término que solo tiene coeficiente lineal, añade provisionalmente su cuadrado al modelo *ya ajustado* y comprueba si esa ampliación reduce el error de ajuste lo suficiente frente al modelo original. También añade el cuadrado del valor ajustado: esta última prueba es el **test de no aditividad de Tukey**, que busca una curvatura global no recogida por la forma actual.

Los términos que ya tienen grado 2 se dibujan, pero **no se vuelven a probar añadiendo el mismo cuadrado**. En la tabla, un p-valor pequeño señala evidencia contra la forma considerada en esa comparación, no identifica por sí solo la mejor transformación. Como miramos varias variables y pruebas, evita leer cada p-valor aislado como una decisión definitiva; comprueba también los gráficos y el sentido de la modificación.

In [ ]:
def residuos_por_termino(modelo, tabla, grupos):
    variables = list(grupos)
    paneles = [(variable, tabla[variable], len(grupos[variable]) == 1) for variable in variables]
    paneles.append(("Ajustados (Tukey)", modelo.fittedvalues, True))
    filas = int(np.ceil(len(paneles) / 3))
    fig, axes = plt.subplots(filas, 3, figsize=(18, 4.5 * filas), squeeze=False)
    resultados = []
    for ax, (nombre, valores, probar) in zip(axes.flat, paneles):
        dibujar_relacion(ax, valores, modelo.resid, nombre)
        ax.set(title=nombre, ylabel="Residuos")
        ax.axhline(0, color="#666666", linestyle="--", linewidth=1.2)
        ax.title.set_fontsize(18)
        ax.xaxis.label.set_fontsize(14)
        ax.yaxis.label.set_fontsize(14)
        ax.tick_params(labelsize=12)
        if probar:
            x = np.asarray(valores, dtype=float)
            x = (x - x.mean()) / x.std()
            ampliada = np.column_stack([modelo.model.exog, x ** 2])
            ampliado = sm.OLS(modelo.model.endog, ampliada).fit()
            f, p, _ = ampliado.compare_f_test(modelo)
            resultados.append({"Término": nombre, "F": f, "p-valor": p})
    for ax in list(axes.flat)[len(paneles):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()
    return pd.DataFrame(resultados).set_index("Término")

In [ ]:
residuos_por_termino(mod_lm2, datos, {v: grupos_lineales[v] for v in seleccion_lineal})

## Transformaciones polinómicas de las predictoras

Permitimos curvatura de grado 2 para `Cement`, `BlastFurnaceSlag`, `FlyAsh`, `Water`, `Superplasticizer`, `FineAggregate` y `Age`; `CoarseAggregate` permanece lineal. En lugar de elevar directamente $x$ al cuadrado, calculamos $z=(x-\bar x_{\mathrm{train}})/s_{\mathrm{train}}$ y creamos las columnas $z$ y $z^2$. Centrar y escalar ayuda a la estabilidad numérica y modifica la interpretación individual de los coeficientes, **no** el conjunto de curvas cuadráticas que puede representar el modelo completo.

`diseno_polinomico(datos)` aprende medias y escalas **solo con entrenamiento** y las guarda. Cuando lleguemos al test, pasaremos esos mismos parámetros: usar su propia media filtraría información del test al preprocesamiento y haría incoherentes las columnas del modelo. `grupos_polinomicos` vincula las dos columnas de cada variable para conservarlas o retirarlas juntas.

Primero ajustamos `mod_lm4` con todos los términos y luego aplicamos la misma eliminación por AIC para obtener `mod_lm5`. Lee `pasos_polinomicos` para seguir cada retirada y el resumen posterior para comprobar el modelo resultante. La inclusión de $z^2$ describe curvatura **manteniendo las otras columnas del modelo constantes**; no implica extrapolación fiable fuera de los rangos vistos.

In [ ]:
def diseno_polinomico(tabla, parametros=None):
    # Aprender medias y escalas solo al ajustar; reutilizarlas para predecir.
    if parametros is None:
        parametros = {v: (tabla[v].mean(), tabla[v].std(ddof=0))
                      for v in predictores if v != "CoarseAggregate"}
    matriz = pd.DataFrame({"const": 1.0}, index=tabla.index)
    for variable in predictores:
        if variable == "CoarseAggregate":
            matriz[variable] = tabla[variable]
        else:
            centro, escala = parametros[variable]
            z = (tabla[variable] - centro) / escala
            matriz[variable + "_1"] = z
            matriz[variable + "_2"] = z ** 2
    return matriz, parametros

In [ ]:
X_polinomico, parametros_polinomicos = diseno_polinomico(datos)
grupos_polinomicos = {
    variable: [variable] if variable == "CoarseAggregate" else [variable + "_1", variable + "_2"]
    for variable in predictores
}
mod_lm4 = sm.OLS(y, X_polinomico).fit()
mod_lm4.summary()

In [ ]:
mod_lm5, seleccion_polinomica, pasos_polinomicos = seleccion_atras(y, X_polinomico, grupos_polinomicos)
pasos_polinomicos

In [ ]:
mod_lm5.summary()

#### Microejercicio práctico — construir un término cuadrático

Usa `Cement` y los parámetros guardados en `parametros_polinomicos` para reconstruir su columna cuadrática. Completa los huecos y compara tus primeros valores con `X_polinomico['Cement_2']`; así compruebas cómo se construye una predictora polinómica sin rehacer todo el diseño.

In [ ]:
try:
    # Reconstruye la columna cuadrática estandarizada de Cement.
    centro_cement, escala_cement = parametros_polinomicos[...]  # Completa el nombre del predictor en el diccionario.
    z_cement = (datos['Cement'] - centro_cement) / escala_cement  # Estandariza usando solo los parámetros aprendidos en entrenamiento.
    cement_cuadrado = ...  # Completa la potencia para obtener el término de segundo grado.
    print(cement_cuadrado.head())  # Inspecciona algunos valores calculados.
    print(X_polinomico['Cement_2'].head())  # Compáralos con la columna creada por diseno_polinomico.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Colinealidad de los términos seleccionados

Dos o más columnas pueden aportar información muy parecida. Aunque el modelo prediga razonablemente, esa **colinealidad** puede volver inestables los coeficientes y ampliar su incertidumbre; es una cuestión distinta de que los residuos muestren curvatura. `gvif` examina la correlación entre **coeficientes estimados** y excluye la constante.

Para términos de un coeficiente, `GVIF` coincide con el VIF habitual y un valor por encima de 5 sirve como **señal para revisar**, no como regla automática de eliminación. Un predictor cuadrático ocupa dos columnas, así que calculamos también `GVIF^(1/(2·Df))` para tener en cuenta los grados de libertad del término; no compares sin más su GVIF bruto con el VIF de un predictor de una sola columna. Revisa qué términos presentan más redundancia antes de interpretar signos y p-valores individuales.

In [ ]:
def gvif(modelo, grupos):
    covarianza = modelo.cov_params().drop(index="const", columns="const")
    desviaciones = np.sqrt(np.diag(covarianza))
    correlacion = covarianza / np.outer(desviaciones, desviaciones)
    columnas = list(correlacion.columns)

    def logdet(matriz):
        if matriz.size == 0:
            return 0.0
        signo, valor = np.linalg.slogdet(matriz)
        if signo <= 0:
            raise ValueError("La matriz de coeficientes es singular.")
        return valor

    total = logdet(correlacion.to_numpy())
    resultados = []
    for variable, terminos in grupos.items():
        dentro = [columnas.index(termino) for termino in terminos]
        fuera = [i for i in range(len(columnas)) if i not in dentro]
        matriz = correlacion.to_numpy()
        valor = np.exp(logdet(matriz[np.ix_(dentro, dentro)]) + logdet(matriz[np.ix_(fuera, fuera)]) - total)
        gl = len(dentro)
        resultados.append({"Término": variable, "GVIF": valor, "Df": gl,
                           "GVIF^(1/(2·Df))": valor ** (1 / (2 * gl))})
    return pd.DataFrame(resultados).set_index("Término")

In [ ]:
grupos_seleccionados = {variable: grupos_polinomicos[variable] for variable in seleccion_polinomica}
gvif(mod_lm5, grupos_seleccionados)

#### Microejercicio práctico — ordenar términos por colinealidad

Calcula el GVIF de los términos del modelo polinómico y ordena la medida `GVIF^(1/(2·Df))` de mayor a menor. Reutiliza la llamada a `gvif` y el ejemplo de `sort_values` de los p-valores; completa el nombre de la columna y el sentido del orden.

In [ ]:
try:
    # Ordena la tabla GVIF por su medida ajustada para identificar los términos más colineales.
    tabla_gvif = gvif(mod_lm5, grupos_seleccionados)  # Calcula los factores para los términos seleccionados.
    orden_gvif = tabla_gvif.sort_values(by=..., ascending=...)  # Completa la columna y el orden descendente.
    display(orden_gvif)  # Presenta los términos desde mayor hasta menor colinealidad ajustada.
    # Usa la tabla para decidir qué término convendría inspeccionar con más detalle.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Validación del modelo polinómico

Repetimos los cuatro diagnósticos y el gráfico de residuos en orden con `mod_lm5`: **la referencia es el modelo lineal seleccionado `mod_lm2`**, no solo el resumen numérico del polinómico. Pregunta: ¿desaparece alguna curvatura en residuos frente a ajustados?, ¿cambia la dispersión?, ¿siguen destacando casos en el Q–Q o el panel de Cook? La corrección de una pauta no implica que todas las hipótesis sean correctas.

La celda siguiente vuelve a trazar residuos frente a los términos conservados y sus posibles pruebas de curvatura. Como muchos ya incluyen $z$ y $z^2$, sus gráficos son más informativos que volver a añadir exactamente ese grado 2. El microejercicio posterior compara medianas de **errores absolutos en entrenamiento**: una medida menor describe estos datos, pero para valorar la capacidad predictiva reservamos el test de la parte final.

In [ ]:
diagnosticos(mod_lm5, "Diagnóstico del modelo polinómico seleccionado")
grafico_orden_residuos(mod_lm5, "Residuos del modelo polinómico en el orden de las filas")

In [ ]:
residuos_por_termino(mod_lm5, datos, grupos_seleccionados)

**Microejercicio — errores de ajuste.** Compara la mediana de los residuos **en valor absoluto** de `mod_lm2` y `mod_lm5`. Reutiliza `abs()` del ejercicio de residuos extremos y el valor `50%` de `describe()`, mostrado al principio. ¿Cuál modelo ajusta mejor estas observaciones? Recuerda que ambas medidas corresponden al entrenamiento, no al test.

In [ ]:
try:
    # Usa el valor 50 % del resumen descriptivo para obtener la mediana de cada magnitud residual.
    mediana_lm2 = mod_lm2.resid.abs().describe()["50%"]  # Reutiliza abs() del ejercicio de residuos extremos.
    mediana_lm5 = ...  # Repite el patrón anterior sustituyendo mod_lm2 por mod_lm5.
    print(f'mod_lm2: {mediana_lm2:.3f}; mod_lm5: {mediana_lm5:.3f}')  # Muestra ambos valores para compararlos.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Transformación Box–Cox de la respuesta

Si la dispersión de los residuos depende del nivel de la respuesta o sus colas se alejan de la normalidad, podemos **explorar** una transformación de `Strength`. Box–Cox requiere valores estrictamente positivos y considera potencias según $\lambda$: $\lambda=1$ corresponde a la escala original y $\lambda=0$ al logaritmo.

El cálculo de la log-verosimilitud perfilada está aislado en `logverosimilitud_boxcox`, importada desde `s3_logverosimilitud_boxcox.py`. Conceptualmente, para cada $\lambda$ transforma la respuesta, vuelve a estimar el modelo con las mismas predictoras y evalúa el ajuste incluyendo el cambio de escala de la transformación. La celda siguiente busca el máximo y construye el perfil; nos centraremos en interpretar el lambda óptimo y los valores plausibles, no en revisar la fórmula línea por línea.

La línea horizontal del perfil marca un corte aproximado del 95 %; observa si $\lambda=1$ queda entre las potencias plausibles. Este análisis es exploratorio: no demuestra que la transformación mejore la predicción, y los coeficientes o el AIC con respuesta transformada no se comparan directamente con el modelo en escala original.

In [ ]:
from s3_logverosimilitud_boxcox import logverosimilitud_boxcox  # Importa el cálculo estadístico del perfil Box–Cox.

# Busca el lambda que maximiza la log-verosimilitud perfilada del modelo seleccionado.
optimo_lambda = minimize_scalar(lambda valor: -logverosimilitud_boxcox(valor, mod_lm5),
                                bracket=(-1, 1), method="brent")
if not optimo_lambda.success:
    raise RuntimeError("No se ha podido estimar lambda.")
lamb = optimo_lambda.x

# Evalúa el perfil en una rejilla para visualizarlo y compararlo con lambda=1.
bc = pd.DataFrame({"lambda": np.linspace(-2, 2, 201)})
bc["logLik"] = [logverosimilitud_boxcox(valor, mod_lm5) for valor in bc["lambda"]]
lamb

In [ ]:
def explorar_boxcox(valor_lambda):
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.plot(bc["lambda"], bc["logLik"], color="#34699a", linewidth=2)
    ax.axvline(lamb, color="#800000", linestyle="--", linewidth=1.8,
               label=f"Lambda estimada: {lamb:.3f}")
    ax.axvline(valor_lambda, color="#6f9fc8", linewidth=1.8,
               label=f"Lambda seleccionada: {valor_lambda:.2f}")
    corte = logverosimilitud_boxcox(lamb, mod_lm5) - stats.chi2.ppf(0.95, 1) / 2
    ax.axhline(corte, color="#666666", linestyle=":", linewidth=1.5,
               label="Corte del intervalo del 95 %")
    ax.set(xlabel="Lambda", ylabel="Log-verosimilitud", title="Perfil Box–Cox")
    ax.legend(fontsize=12)
    fig.tight_layout()
    plt.show()

In [ ]:
# Compara el perfil con la transformación identidad (lambda = 1).
explorar_boxcox(1.0)

#### Microejercicio práctico — explorar una transformación logarítmica

Reutiliza la llamada `explorar_boxcox(1.0)` de la celda anterior y cambia la lambda marcada para representar el logaritmo ($\lambda=0$). Usa el gráfico para comprobar si ese valor queda por encima del corte del 95 %.

In [ ]:
try:
    lambda_prueba = ...  # Completa con lambda = 0, que corresponde a la transformación logarítmica.
    explorar_boxcox(lambda_prueba)  # Dibuja el perfil con el valor candidato marcado.
except Exception as error:
    plt.close("all")  # Evita dejar un gráfico parcial si el valor no es válido.
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Ajustar un modelo con la respuesta transformada

Con la estimación `lamb` creamos `Strength2`. La función usa $\log(y)$ cuando $\lambda$ es prácticamente cero y, en otro caso, $y^\lambda$. Para un $\lambda$ fijo distinto de cero, esta potencia y la forma Box–Cox estándar $(y^\lambda-1)/\lambda$ difieren en un cambio afín; el perfil anterior sirve para escoger la potencia, mientras este modelo se ajusta con la forma concreta indicada en el código.

`mod_lm6` conserva los términos polinómicos de las otras predictoras, pero **excluye `CoarseAggregate`** en este ajuste ilustrativo. Su respuesta ahora es `Strength2`: un cambio unitario de una predictora modifica la *escala transformada*, no directamente los MPa originales. Por ello, los coeficientes, residuos y AIC de `mod_lm6` **no deben compararse como si la respuesta fuera idéntica** a la de `mod_lm5`. Esta rama no sustituye al modelo final de predicción en la escala original.

In [ ]:
def transformar_respuesta(valores, valor_lambda):
    return np.log(valores) if abs(valor_lambda) < 1e-8 else np.power(valores, valor_lambda)

datos["Strength2"] = transformar_respuesta(datos["Strength"], lamb)
variables_sin_coarse = [variable for variable in predictores if variable != "CoarseAggregate"]
columnas_sin_coarse = ["const"] + [columna for variable in variables_sin_coarse
                                   for columna in grupos_polinomicos[variable]]
mod_lm6 = sm.OLS(datos["Strength2"], X_polinomico[columnas_sin_coarse]).fit()
mod_lm6.summary()

## Validación del modelo con respuesta transformada

Aplicamos la misma función de cuatro paneles a `mod_lm6`. Busca si la curva de residuos frente a ajustados se acerca a cero, si la dispersión se hace más uniforme y si las colas del Q–Q siguen desviándose. La transformación podría mejorar un aspecto y no otro: **no basta con mirar un indicador**.

Ten presente que ahora el residuo es `Strength2` observada menos `Strength2` ajustada; la amplitud de sus ejes no está en las mismas unidades que los residuos de `mod_lm5`. Lo útil es comparar la **forma** de los patrones, no el tamaño numérico de los residuos sin volver a la escala original.

In [ ]:
diagnosticos(mod_lm6, "Diagnóstico del modelo con Strength2")

### Ejercicio 3.1 — Transformar predictores con cuidado

Este ejercicio cambia **una predictora**, no la respuesta: mantenemos `Strength` para poder comparar modelos ajustados sobre **las mismas filas y la misma escala**. Trabaja sobre una copia y conserva `datos`, `X_lineal` y los modelos anteriores; las celdas posteriores los necesitan.

1. **Comprobar el dominio.** Examina los mínimos de las predictoras originales. $\log(x)$ solo está definido para $x>0$ en estos cálculos: detecta las columnas que contienen ceros y elige `Cement`, positiva en todo el entrenamiento. No uses `np.log` directamente sobre columnas con cero.
2. **Construir y ajustar.** Empieza por `X_lineal.copy()`, sustituye **solo** su columna `Cement` por `np.log` de los valores correspondientes y ajusta un nuevo `sm.OLS(y, ...)`. Compara su AIC con `mod_lm1.aic`: las filas y la respuesta son las mismas, de modo que esa comparación es pertinente. Un AIC más bajo sugiere una mejor combinación de ajuste y complejidad, no prueba que el modelo sea correcto.
3. **Diagnosticar.** Usa `diagnosticos(...)` con el nuevo ajuste y compáralo visualmente con el modelo lineal original: busca cambios en curvatura, dispersión, Q–Q e influencia. ¿La transformación mejora la pauta que te preocupaba o aparece otro problema?

Escribe cada apartado en su celda de pistas. No sustituyas `mod_lm5`: sigue siendo el modelo final que se usará para evaluar test.

In [ ]:
try:
    # 3.1.1 — Comprobar el dominio de las transformaciones.
    # Pista: consulta los mínimos de predictores; un valor cero impide usar np.log directamente.
    # Tu código:
    pass
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

In [ ]:
try:
    # 3.1.2 — Ajustar un modelo con el logaritmo de Cement.
    # Pista: parte de X_lineal.copy(), modifica solo la copia y ajusta con sm.OLS(y, ...).
    # Compara su AIC con mod_lm1.aic; no sobrescribas los modelos originales.
    # Tu código:
    pass
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

In [ ]:
try:
    # 3.1.3 — Diagnosticar el modelo alternativo.
    # Pista: reutiliza diagnosticos(...) y contrasta los patrones con el modelo sin logaritmos.
    # Tu código:
    pass
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

## Observaciones influyentes

Un **residuo grande** significa que una respuesta está lejos de la predicción; **apalancamiento alto** significa que la combinación de predictoras es inusual; **influencia alta** significa que una observación puede alterar bastante el modelo ajustado. No son sinónimos y un caso puede sobresalir en una medida sin hacerlo en las otras.

Para `mod_lm5` calculamos residuos studentizados externos, apalancamiento y distancia de Cook. Los p-valores de los residuos extremos se ajustan por Bonferroni al examinar muchas observaciones: es una forma conservadora de reducir falsos positivos. En el panel correspondiente dibujamos $-\log_{10}(p)$; los puntos **por encima** de la línea roja tienen p ajustado menor que 0,05, y p ajustado igual a 1 se sitúa en cero. Las referencias verticales del diagrama de burbujas corresponden a $2p/n$ y $3p/n$ (con $p$ parámetros), no a umbrales absolutos de error.

**Siguiente paso:** identificar y examinar los casos señalados en sus variables originales. Ni un p pequeño ni una burbuja grande prueban que el dato sea erróneo; no elimines filas automáticamente.

In [ ]:
with threadpool_limits(limits=1):
    influencia_modelo = mod_lm5.get_influence()
    studentizados = influencia_modelo.resid_studentized_external

influencia = pd.DataFrame({
    "Cook": influencia_modelo.cooks_distance[0],
    "Residuo studentizado": studentizados,
    "p Bonferroni": np.minimum(1, 2 * stats.t.sf(np.abs(studentizados), mod_lm5.df_resid - 1) * len(datos)),
    "Apalancamiento": influencia_modelo.hat_matrix_diag,
}, index=datos.index)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, columna in zip(axes.flat, influencia.columns):
    if columna == "p Bonferroni":
        valores = -np.log10(influencia[columna].clip(lower=np.finfo(float).tiny))
        significativos = influencia[columna] < 0.05
        _ = ax.scatter(influencia.index[~significativos], valores[~significativos],
                       s=30, color="#9fc3e4", edgecolor="#666666", linewidth=0.4, alpha=0.65)
        if significativos.any():
            _ = ax.scatter(influencia.index[significativos], valores[significativos],
                           s=60, color="#800000", edgecolor="white", linewidth=0.6,
                           label="p ajustado < 0,05")
        _ = ax.axhline(-np.log10(0.05), color="#800000", linestyle="--", linewidth=1.8,
                       label="Umbral: p ajustado = 0,05")
        _ = ax.set_ylabel("−log₁₀(p Bonferroni)")
        _ = ax.legend(fontsize=10)
    else:
        _ = ax.vlines(influencia.index, 0, influencia[columna], color="#6f9fc8", alpha=0.7)
        valores = influencia[columna]
        _ = ax.set_ylabel(columna)
    _ = ax.set(xlabel="Observación", title=columna)
    for observacion in [65, 81, 147, 248]:
        if columna == "p Bonferroni" and influencia.loc[observacion, columna] == 1:
            continue  # Los p-valores iguales a 1 quedan juntos en cero.
        desplazamiento = (-14, 8) if columna == "Apalancamiento" and observacion == 65 else (4, 5)
        alineacion = "right" if desplazamiento[0] < 0 else "left"
        _ = ax.annotate(str(observacion), (observacion, valores.loc[observacion]),
                        xytext=desplazamiento, textcoords="offset points", fontsize=10,
                        ha=alineacion, color="#800000")
    ax.title.set_fontsize(18)
    ax.xaxis.label.set_fontsize(16)
    ax.yaxis.label.set_fontsize(16)
    ax.tick_params(labelsize=14)
fig.tight_layout()
plt.show()

**Microejercicio — observaciones influyentes.** Reutiliza el método `.nlargest()` del ejercicio anterior y aplícalo a la columna `Cook` de `influencia` para identificar las tres observaciones con mayor distancia. ¿Coinciden necesariamente con las de mayor residuo?

In [ ]:
try:
    # Lista las tres observaciones con mayor distancia de Cook.
    cook_mayores = influencia['Cook'].nlargest(...)  # Completa el número de casos que quieres inspeccionar.
    print(cook_mayores)  # Muestra los índices y valores de Cook, de mayor a menor.
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa los huecos y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

### Residuos, apalancamiento e influencia

Cada burbuja es una observación: el eje horizontal muestra su **apalancamiento**, el vertical su **residuo studentizado externo** y el tamaño representa la **distancia de Cook relativa al máximo observado**. Los trazos horizontales en $\pm2$ ayudan a localizar residuos llamativos; los verticales en $2p/n$ y $3p/n$ son referencias heurísticas de apalancamiento. Ninguna línea decide por sí sola qué hacer con un dato.

Activa las etiquetas y compara las observaciones 81 y 147, la 248 con residuo destacado y las 65 y 81 por su apalancamiento. Pregunta **dónde** está cada caso y **qué combinación** de residuo y apalancamiento explica el tamaño de su burbuja; después comprueba los valores reales en la tabla.

In [ ]:
def grafico_influencia(etiquetas):
    fig, ax = plt.subplots(figsize=(16, 8))
    tamanos = 20 + 800 * influencia["Cook"] / influencia["Cook"].max()
    ax.scatter(influencia["Apalancamiento"], influencia["Residuo studentizado"],
               s=tamanos, color="#9fc3e4", alpha=0.65,
               edgecolors="#666666", linewidth=0.6)
    ax.axhline(0, color="#800000", linestyle="--", linewidth=1.8)
    for valor in [-2, 2]:
        ax.axhline(valor, color="#666666", linestyle=":", linewidth=1.2)
    for factor in [2, 3]:
        ax.axvline(factor * len(mod_lm5.params) / len(datos), color="#800000",
                   linestyle=":", linewidth=1.5)
    if etiquetas:
        for observacion in [65, 81, 147, 248]:
            fila = influencia.loc[observacion]
            ax.annotate(str(observacion), (fila["Apalancamiento"], fila["Residuo studentizado"]),
                        xytext=(6, 5), textcoords="offset points", fontsize=12,
                        color="#800000")
    ax.set(xlabel="Apalancamiento", ylabel="Residuo studentizado",
           title="Influencia de las observaciones")
    fig.tight_layout()
    plt.show()

In [ ]:
# True muestra las etiquetas de las observaciones destacadas.
grafico_influencia(True)

#### Inspeccionar las observaciones seleccionadas

`obs_rm` reúne las etiquetas 81, 147 y 248 para examinarlas con `datos.loc[obs_rm]`. Revisa las predictoras **y** la respuesta: ¿hay una combinación inusual, una edad extrema o un valor que podría ser una errata? `loc` busca etiquetas de fila, no posiciones; por eso coincide con los números señalados en los gráficos.

Aunque se llamen `obs_rm` en el código, **aún no se eliminan de `datos`**. Primero comprobamos su posición en todas las relaciones bivariantes. La observación 65 se muestra en el gráfico de influencia por su apalancamiento, pero no forma parte de esta lista de tres casos para el reajuste ilustrativo.

In [ ]:
obs_rm = [81, 147, 248]
datos.loc[obs_rm]

#### Localizarlas en los gráficos de pares

En una nueva matriz de dispersión coloreamos y etiquetamos las observaciones 81, 147 y 248, mientras el resto queda en segundo plano. Esto ayuda a entender si cada caso es inusual **en una variable concreta** o solo en la **combinación de varias**; un punto puede parecer ordinario en cada gráfico por separado y aun así tener mucho apalancamiento multivariable.

Compara las etiquetas con `datos.loc[obs_rm]`. El color sirve únicamente para encontrarlas; no significa que se haya confirmado un error de medición ni que convenga borrarlas del análisis.

In [ ]:
col_points = np.array(["#9fc3e433"] * len(datos), dtype=object)
tamanos = np.full(len(datos), 16.0)
colores_destacados = ["#800000", "#34699a", "#6f9fc8"]
for observacion, color in zip(obs_rm, colores_destacados):
    posicion = datos.index.get_loc(observacion)
    col_points[posicion] = color
    tamanos[posicion] = 85  # Destacan sin tapar los demás puntos.

axes = scatter_matrix(datos[columnas_originales], figsize=(18, 18), diagonal="hist",
                      c=col_points, s=tamanos, alpha=None,
                      hist_kwds={"color": "#9fc3e4", "edgecolor": "#666666"})
for i, variable_y in enumerate(columnas_originales):
    for j, variable_x in enumerate(columnas_originales):
        eje = axes[i, j]
        eje.tick_params(labelsize=9)
        eje.xaxis.label.set_fontsize(11)
        eje.yaxis.label.set_fontsize(11)
        if i != j:
            for observacion, color in zip(obs_rm, colores_destacados):
                _ = eje.annotate(str(observacion),
                                 (datos.loc[observacion, variable_x], datos.loc[observacion, variable_y]),
                                 xytext=(3, 3), textcoords="offset points", fontsize=8, color=color)
plt.show()

### Ajustar sin las observaciones señaladas: análisis de sensibilidad

Creamos **una copia** `datos_rm` excluyendo 81, 147 y 248; los datos originales quedan intactos. `mod_lm7` se reajusta sin ellas con las siete predictoras polinómicas utilizadas aquí, sin `CoarseAggregate`. El diseño vuelve a calcular medias y escalas en la muestra reducida, como corresponde a **un nuevo ajuste**. La celda siguiente vuelve a mostrar el resumen de `mod_lm5` para comparar estabilidad de coeficientes y resultados.

**Precaución:** `mod_lm7` no es un simple `mod_lm5` sin tres residuos: además de quitar filas, cambia los términos (excluye `CoarseAggregate`). Por tanto, una diferencia entre resúmenes **no puede atribuirse solo** a la eliminación de esas observaciones. Este apartado ilustra sensibilidad del ajuste, no una prueba de que debamos descartarlas; el modelo final conserva la muestra completa.

In [ ]:
datos_rm = datos.drop(index=obs_rm).copy()
X_rm, _ = diseno_polinomico(datos_rm)
mod_lm7 = sm.OLS(datos_rm["Strength"], X_rm[columnas_sin_coarse]).fit()
mod_lm7.summary()

In [ ]:
mod_lm5.summary()

### Modelo final

Asignamos `mod_final = mod_lm5`: el modelo polinómico seleccionado **con todas las observaciones de entrenamiento** y respuesta `Strength` en la escala original. Los modelos con respuesta transformada y con filas excluidas fueron análisis complementarios; no los usamos para predecir en test.

Este punto fija la decisión **antes de consultar los resultados de test**. A partir de aquí se puede interpretar el modelo y comprobar cuánto error comete en datos nuevos, pero no cambiar su especificación solo porque el test resulte mejor o peor: eso convertiría la evaluación en parte del entrenamiento.

In [ ]:
mod_final = mod_lm5

### Efectos de cada predictor en el modelo final

Para visualizar cada relación ajustada, `datos_efecto(variable)` crea una rejilla de 100 valores entre el mínimo y el máximo observados de esa predictora. Forma nuevas filas del **diseño del modelo** variando las columnas lineal y cuadrática de esa variable y dejando las otras columnas en sus medias del entrenamiento; luego obtiene la predicción media y su **intervalo de confianza del 95 %** mediante `get_prediction`.

La línea muestra la resistencia media estimada en este escenario y la banda, la incertidumbre de esa **media**, no el rango esperado de una pieza individual (eso requeriría un intervalo de predicción). Las columnas cuadráticas de las otras variables se fijan en sus propias medias, que no equivalen necesariamente al cuadrado de una media: la combinación dibujada es un **perfil de diseño** y puede no coincidir exactamente con una receta de hormigón real. Evita interpretarlo como efecto causal.

La cuadrícula enseña todos los términos seleccionados; debajo puedes cambiar `variable` o desactivar `banda` para examinar uno con más detalle. No extrapoles estas curvas más allá del rango mostrado.

In [ ]:
def datos_efecto(variable):
    valores = np.linspace(datos[variable].min(), datos[variable].max(), 100)
    columnas = mod_final.model.exog_names
    medias = mod_final.model.exog.mean(axis=0)
    matriz = pd.DataFrame(np.tile(medias, (len(valores), 1)), columns=columnas)
    if variable in parametros_polinomicos:
        centro, escala = parametros_polinomicos[variable]
        z = (valores - centro) / escala
        matriz[grupos_polinomicos[variable]] = np.column_stack((z, z ** 2))
    else:
        matriz[variable] = valores
    resumen = mod_final.get_prediction(matriz).summary_frame(alpha=0.05)
    return valores, resumen

def dibujar_efecto(ax, variable, banda=True):
    valores, resumen = datos_efecto(variable)
    if banda:
        ax.fill_between(valores, resumen["mean_ci_lower"].to_numpy(), resumen["mean_ci_upper"].to_numpy(),
                        color="#9fc3e4", alpha=0.5, label="IC de la media (95 %)")
    ax.plot(valores, resumen["mean"].to_numpy(), color="#800000", linewidth=1.8)
    ax.set(xlabel=variable, ylabel="Strength predicha", title=variable)

In [ ]:
filas_efectos = int(np.ceil(len(seleccion_polinomica) / 4))
fig, axes = plt.subplots(filas_efectos, 4, figsize=(20, 5 * filas_efectos), squeeze=False)
for ax, variable in zip(axes.flat, seleccion_polinomica):
    dibujar_efecto(ax, variable)
    ax.title.set_fontsize(16)
    ax.xaxis.label.set_fontsize(13)
    ax.yaxis.label.set_fontsize(13)
    ax.tick_params(labelsize=11)
for ax in list(axes.flat)[len(seleccion_polinomica):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

Examinamos con más detalle el primer predictor seleccionado por el modelo. Puedes cambiar el predictor o quitar la banda de confianza modificando los argumentos de la llamada siguiente.

In [ ]:
def explorar_efecto(variable, banda):
    fig, ax = plt.subplots(figsize=(16, 8))
    dibujar_efecto(ax, variable, banda)
    fig.tight_layout()
    plt.show()

In [ ]:
# Cambia seleccion_polinomica[0] por otro predictor seleccionado, o True por False.
explorar_efecto(seleccion_polinomica[0], True)

## Evaluar con nuevos datos

### Leer el conjunto de test

Hasta ahora **ajustamos y seleccionamos con entrenamiento**. El archivo de test contiene casos distintos, reservados para comprobar cómo generaliza el modelo elegido. Lo leemos con el mismo separador y numeramos sus filas desde 1 para poder seleccionar un caso mediante `loc`.

El código crea también `Strength2` usando la `lamb` aprendida en entrenamiento, para mantener disponible la transformación estudiada; **`mod_final` no utiliza esa columna**: predice `Strength` en su escala original. No recalcules `lamb` ni el preprocesamiento polinómico con el test y no uses sus valores observados para reajustar el modelo. Su respuesta solo se consultará al representar y cuantificar los errores.

In [ ]:
test = pd.read_csv(DATA_PATH / "Concrete_test.txt", sep="\t")
test.index = np.arange(1, len(test) + 1)
test.index.name = "observacion"
test["Strength2"] = transformar_respuesta(test["Strength"], lamb)

### Construir y obtener las predicciones

`diseno_polinomico(test, parametros_polinomicos)` aplica a cada predictor de test **las medias y escalas guardadas durante el entrenamiento**. El orden de columnas importa: `X_test[mod_final.model.exog_names]` selecciona exactamente las variables que espera el modelo final y en el mismo orden. Después, `predict` calcula una resistencia estimada por observación; `pr` conserva el índice de test para emparejar cada predicción con su respuesta real.

El modelo aún **no ha usado `Strength` de test** en esta operación. Comprueba el número y el rango de las predicciones: valores extraños pueden revelar entradas fuera del dominio de entrenamiento o un error de preprocesamiento, no necesariamente un fallo del algoritmo de ajuste.

In [ ]:
X_test, _ = diseno_polinomico(test, parametros_polinomicos)
pr = mod_final.predict(X_test[mod_final.model.exog_names]).rename("Prediccion")
pr

### Predicciones y valores observados: un caso a la vez

Cada punto del gráfico compara una **predicción de test (eje horizontal)** con la resistencia observada (eje vertical). La diagonal representa coincidencia exacta: por encima, el modelo **subestimó** la resistencia; por debajo, la **sobreestimó**. La observación elegida destaca en granate y la tabla de la derecha muestra sus predictoras originales, su valor real, la predicción y el residuo **real menos predicho**.

Sobre la coordenada horizontal del caso aparecen dos intervalos estimados por el modelo entrenado: el **IC del 95 % para la respuesta media** de observaciones con esas predictoras y el **IP del 95 % para una observación individual**. El IP suele ser más ancho porque incorpora la dispersión de las respuestas individuales, además de la incertidumbre sobre su media. **Ninguno se calcula a partir del valor real de test del caso mostrado**; la posición del punto real permite compararlos visualmente, pero que un único punto caiga fuera del IP no invalida el modelo por sí solo.

Desliza el control o escribe la etiqueta de observación para explorar otros casos. Si el widget no aparece en tu editor, ejecuta `explorar_prediccion(5)` cambiando el número.

In [ ]:
from functools import partial
from s3_explorar_prediccion import explorar_prediccion as _explorar_prediccion

# Fija el contexto del modelo y deja la observación como único argumento interactivo.
explorar_prediccion = partial(
    _explorar_prediccion, mod_final, X_test, pr, test, columnas_originales
)

In [ ]:
from ipywidgets import BoundedIntText, HBox, HTML, IntSlider, Layout, VBox, link

selector_predicciones = IntSlider(
    value=5, min=int(test.index.min()), max=int(test.index.max()), step=1,
    description="Observación", continuous_update=False, readout=False,
    style={"description_width": "105px"}, layout=Layout(width="520px"),
)
numero_predicciones = BoundedIntText(
    value=5, min=selector_predicciones.min, max=selector_predicciones.max,
    description="Ir al n.º", style={"description_width": "65px"},
    layout=Layout(width="170px"),
)
enlace_predicciones = link((selector_predicciones, "value"), (numero_predicciones, "value"))
visor_predicciones = HTML(value=explorar_prediccion(selector_predicciones.value).data,
                          layout=Layout(width="100%"))

def actualizar_prediccion(cambio):
    visor_predicciones.value = explorar_prediccion(cambio["new"]).data

selector_predicciones.observe(actualizar_prediccion, names="value")
control_predicciones = VBox([
    HTML("<div style='border-left: 4px solid #800000; padding: 8px 12px; "
         "font-size: 18px; font-weight: 600;'>Explora las predicciones en test</div>"),
    HBox([selector_predicciones, numero_predicciones], layout=Layout(align_items="center")),
    visor_predicciones,
], layout=Layout(width="100%", gap="12px"))
display(control_predicciones)

### Error de predicción en test

Para cada observación, el error de predicción puede escribirse como $\widehat y_i-y_i$; cambiar el signo no afecta a su cuadrado ni a su valor absoluto. El **error cuadrático medio** es $\mathrm{EQM}=\frac1n\sum_i(\widehat y_i-y_i)^2$. En el código, `n` cuenta las filas de test y `pr` y `test["Strength"]` se emparejan por su índice. El EQM penaliza mucho los errores grandes, pero sus unidades son $\mathrm{MPa}^2$, menos intuitivas.

Tomamos la raíz: $\mathrm{RMSE}=\sqrt{\mathrm{EQM}}$, de nuevo en **MPa**. Por ejemplo, un RMSE mayor indica errores cuadráticos típicos mayores en este test; no es la precisión garantizada para cada futura observación. Calculamos aquí estas dos medidas y después reunimos también MAE, MAPE y $R^2$ de test. Ninguna cifra reemplaza la inspección de casos individuales y posibles sesgos sistemáticos.

In [ ]:
n = test.shape[0]
EQM = ((pr - test["Strength"]) ** 2).sum() / n
EQM

In [ ]:
RMSE = np.sqrt(EQM)
RMSE

**Microejercicio — error absoluto en test.** Calcula en una sola línea de código el error absoluto medio (MAE) entre `pr` y `test["Strength"]`. ¿En qué unidades se expresa? Compáralo con `RMSE`, que da más peso a los errores grandes.

In [ ]:
# Calcula el error absoluto medio de las predicciones sobre el conjunto de test.
error_absoluto = (pr - test['Strength']).abs()  # Calcula la magnitud del error fila por fila.
MAE = error_absoluto.mean()  # Completa el agregado que resume el error absoluto medio.
print(f'MAE en test: {MAE:.2f} MPa')  # Muestra el error en las mismas unidades que Strength.

### Resumen de métricas en test

Comprueba tu MAE del microejercicio con la tabla: $\mathrm{MAE}=\frac1n\sum_i|\widehat y_i-y_i|$ también se expresa en **MPa**, pero crece menos que el RMSE ante errores muy grandes. **Antes de ejecutar la celda de resumen**, completa la celda del microejercicio para definir `MAE` si has reiniciado el kernel.

El **MAPE** promedia $100|\widehat y_i-y_i|/|y_i|$ y se expresa en porcentaje. Solo tiene sentido cuando las respuestas observadas son distintas de cero; el código devuelve `NaN` si alguna vale cero, y cerca de cero puede resultar engañoso. El **$R^2$ de test** compara la suma de errores cuadrados del modelo con la de predecir siempre la media observada de este test **solo como referencia de evaluación**. Puede ser **negativo** si el modelo empeora esa referencia; no es un parámetro que se ajuste con el test.

Para contextualizar RMSE y MAE, las dos celdas siguientes muestran la desviación típica y los cuantiles de `Strength` observada en test. Son descripciones de esa muestra, no puntuaciones de calidad del modelo. No uses estas métricas para volver atrás y elegir otro modelo con el mismo test.

In [ ]:
observados = test["Strength"]
errores = pr - observados
MAPE = np.nan if (observados == 0).any() else (100 * errores.abs() / observados.abs()).mean()
R2_test = 1 - (errores ** 2).sum() / ((observados - observados.mean()) ** 2).sum()
pd.Series({"EQM": EQM, "RMSE": RMSE, "MAPE (%)": MAPE, "R² test": R2_test},
          name="Evaluación en test").to_frame()

La desviación típica observada describe la variabilidad de la resistencia antes de usar las predicciones.

In [ ]:
test["Strength"].std(ddof=1)

In [ ]:
test["Strength"].describe()

### Ejercicio 3.2 — Comparar predicciones en test

Ya conocemos el `RMSE` del modelo final sobre observaciones que no se utilizaron en su ajuste. Compáralo con un **referente sencillo previamente entrenado**, `mod_lm0`, sin cambiar los modelos ni `pr`:

1. Construye para `test` la misma matriz que usó `mod_lm0`: columna constante y `Cement`, en ese orden. Pasa esa matriz a `mod_lm0.predict(...)` y guarda las predicciones con **otro nombre**. No incluyas los demás predictores, porque ese modelo no los conoce.
2. Calcula el RMSE de esas nuevas predicciones frente a `test["Strength"]` y compáralo con `RMSE`. Ambos errores estarán en **MPa** y se medirán sobre las **mismas filas**: así la comparación responde a cuál predice mejor en este conjunto de test.

¿Por qué no basta con comparar el $R^2$ de entrenamiento? Porque mide ajuste en los datos usados para estimar los coeficientes, no error en observaciones nuevas. **No reajustes ni selecciones** modelos según este test: repetir elecciones sobre él haría optimista la evaluación. Si el modelo más complejo no gana aquí, piensa en sobreajuste, incertidumbre y el tamaño de esta muestra antes de generalizar la conclusión.

In [ ]:
try:
    # 3.2.1 — Predecir la resistencia de test con el modelo simple.
    # Pista: construye la misma matriz con constante y Cement que en X_simple.
    # Guarda las nuevas predicciones con otro nombre; conserva pr sin cambios.
    # Tu código:
    pass
except Exception as error:
    print("El ejercicio no está bien implementado todavía. Revisa tu código y vuelve a intentarlo.")
    print(f"Detalle: {type(error).__name__}: {error}")

In [ ]:
# 3.2.2 — Comparar el error cuadrático medio en test.
# Pista: calcula el RMSE de las nuevas predicciones y compáralo con RMSE del modelo final.
# No reajustes ni elijas modelos basándote en este conjunto de test.
# Tu código:

### Documentación de las herramientas

- [Regresión por mínimos cuadrados en statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).
- [Suavizado LOWESS](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html).
- [Transformación Box–Cox en SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.boxcox.html).